# NamOS Trainer

**Proprietary Training Methodology**

This is proprietary training methodology developed by Tonkraf. The code is provided for your personal use in training models compatible with the NamOS plugin. The underlying training method is proprietary.

**Upload order:**
1. DI file (input.wav)
2. Pre-trained .nam models
3. Capture .wav files


In [ ]:
!pip install -q torch librosa soxr numpy tqdm
print("✓ Dependencies installed")


In [ ]:
from google.colab import files
import os, shutil
for d in ['pretrained', 'captures', 'di', 'output']:
    os.makedirs(d, exist_ok=True)
print("📁 Upload your DI file (input.wav):")
uploaded_di = files.upload()
for name, data in uploaded_di.items():
    with open(f'di/{name}', 'wb') as f:
        f.write(data)
    print(f"  ✓ Saved di/{name}")
print("\n📁 Upload pre-trained .nam models:")
uploaded_nams = files.upload()
for name, data in uploaded_nams.items():
    with open(f'pretrained/{name}', 'wb') as f:
        f.write(data)
    print(f"  ✓ Saved pretrained/{name}")
print("\n📁 Upload capture .wav files:")
uploaded_captures = files.upload()
for name, data in uploaded_captures.items():
    with open(f'captures/{name}', 'wb') as f:
        f.write(data)
    print(f"  ✓ Saved captures/{name}")
print("\n✓ All files uploaded!")


In [ ]:
import ipywidgets as widgets
from IPython.display import display
epochs_w = widgets.IntSlider(value=200, min=50, max=500, step=10, description='Epochs:')
patience_w = widgets.IntSlider(value=30, min=10, max=100, step=5, description='Patience:')
lr_w = widgets.FloatLogSlider(value=0.0001, base=10, min=-5, max=-2, step=0.5, description='LR:')
batch_w = widgets.Dropdown(options=[16, 32, 64, 128], value=64, description='Batch:')
ny_w = widgets.Dropdown(options=[4096, 8192, 16384], value=8192, description='Chunk size:')
val_freq_w = widgets.IntSlider(value=5, min=1, max=20, description='Val freq:')
steps_per_epoch_w = widgets.IntSlider(value=200, min=50, max=500, step=10, description="Steps/epoch:")
display(widgets.VBox([
    widgets.HTML("<h3>Training Hyperparameters</h3>"),
    epochs_w, patience_w, lr_w, batch_w, ny_w, val_freq_w, steps_per_epoch_w
]))
print("👆 Adjust settings, then run the next cell to train.")


In [ ]:
import os, sys, json, math, time, re, warnings
from pathlib import Path
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import librosa
import soxr
from tqdm.notebook import tqdm
warnings.filterwarnings("ignore")
TRAIN_RATES = [96000, 192000]
MAX_RATE = 384000.0
def resample_audio(audio, orig_sr, target_sr):
    return soxr.resample(audio, orig_sr, target_sr, quality="VHQ")
_di_cache = {}
def get_resampled_di(di_path, orig_sr, target_sr):
    key = (str(di_path), target_sr)
    if key not in _di_cache:
        di_orig, _ = librosa.load(str(di_path), sr=None, mono=True)
        _di_cache[key] = resample_audio(di_orig, orig_sr, target_sr)
    return _di_cache[key]
class Conv1d(nn.Conv1d):
    def forward(self, x):
        return super().forward(x)
class RateCondition(nn.Module):
    def __init__(self, num_biases, hidden=16):
        super().__init__()
        self.fc1 = nn.Linear(1, hidden)
        self.fc2 = nn.Linear(hidden, num_biases)
        nn.init.zeros_(self.fc2.weight)
        nn.init.zeros_(self.fc2.bias)
    def forward(self, rate_norm):
        h = torch.tanh(self.fc1(rate_norm))
        return self.fc2(h)
class Layer(nn.Module):
    def __init__(self, condition_size, channels, kernel_size, dilation,
                 activation="Tanh", gated=False):
        super().__init__()
        self.gated = gated
        mid_channels = 2 * channels if gated else channels
        self.conv = Conv1d(channels, mid_channels, kernel_size,
                           dilation=dilation, padding=0)
        self.input_mixer = Conv1d(condition_size, mid_channels, 1, bias=False)
        self._1x1 = Conv1d(channels, channels, 1)
        self.activation = nn.Tanh()
    def forward(self, x, h, out_length, rate_bias=None):
        z = self.conv(x)
        hmix = self.input_mixer(h)
        hmix = hmix[:, :, -z.shape[2]:]
        z = z + hmix
        if rate_bias is not None:
            z = z + rate_bias.unsqueeze(2)
        if self.gated:
            a, b = z.chunk(2, dim=1)
            z = self.activation(a) * torch.sigmoid(b)
        else:
            z = self.activation(z)
        residual = x[:, :, -z.shape[2]:] + self._1x1(z)
        head_term = z[:, :, -out_length:]
        return residual, head_term
class Layers(nn.Module):
    def __init__(self, input_size, condition_size, head_size, channels,
                 kernel_size, dilations, activation, gated, head_bias):
        super().__init__()
        self.rechannel = Conv1d(input_size, channels, 1, bias=False)
        self.layers = nn.ModuleList([
            Layer(condition_size, channels, kernel_size, d, activation, gated)
            for d in dilations
        ])
        self.head_rechannel = Conv1d(channels, head_size, 1, bias=head_bias)
    @property
    def receptive_field(self):
        k = self.layers[0].conv.kernel_size[0]
        total_dilation = sum(l.conv.dilation[0] for l in self.layers)
        return 1 + (k - 1) * total_dilation
    def forward(self, x, c, head_input=None, rate_biases=None):
        out_length = x.shape[2] - (self.receptive_field - 1)
        if head_input is not None:
            head_input = head_input[:, :, -out_length:]
        x = self.rechannel(x)
        for i, layer in enumerate(self.layers):
            rb = rate_biases[i] if rate_biases is not None else None
            x, ht = layer(x, c, out_length, rate_bias=rb)
            head_input = ht if head_input is None else head_input + ht
        return self.head_rechannel(head_input), x
class WaveNetCore(nn.Module):
    def __init__(self, layers_configs, head_config=None, head_scale=1.0,
                 rate_condition_hidden=16, use_rate_condition=True):
        super().__init__()
        self.layers = nn.ModuleList([Layers(**lc) for lc in layers_configs])
        self.head = None
        if head_config is not None:
            self.head = self._build_head(**head_config)
        self.head_scale = head_scale
        self.use_rate_condition = use_rate_condition
        if use_rate_condition:
            num_biases = sum(
                (2 * lc['channels'] if lc.get('gated', False) else lc['channels'])
                for lc in layers_configs for _ in lc['dilations']
            )
            self.rate_condition = RateCondition(num_biases,
                                                hidden=rate_condition_hidden)
    def _build_head(self, in_channels, channels, out_channels, num_layers):
        layers = []
        for i in range(num_layers):
            ic = in_channels if i == 0 else channels
            oc = out_channels if i == num_layers - 1 else channels
            layers.extend([nn.Tanh() if i > 0 else nn.Identity(),
                          Conv1d(ic, oc, 1)])
        return nn.Sequential(*layers)
    @property
    def receptive_field(self):
        return 1 + sum(l.receptive_field - 1 for l in self.layers)
    def forward(self, x):
        if self.use_rate_condition:
            rate_norm = x[:, 1, 0:1]
            raw_biases = self.rate_condition(rate_norm)
            rate_biases_flat = rate_norm * raw_biases
            bias_idx = 0
            all_biases = []
            for lb in self.layers:
                block_biases = []
                for layer in lb.layers:
                    n = layer.conv.out_channels
                    block_biases.append(rate_biases_flat[:, bias_idx:bias_idx + n])
                    bias_idx += n
                all_biases.append(block_biases)
        else:
            all_biases = [None] * len(self.layers)
        y, head_input = x, None
        for i, layer_block in enumerate(self.layers):
            head_input, y = layer_block(y, x, head_input=head_input,
                                         rate_biases=all_biases[i])
        head_input = self.head_scale * head_input
        if self.head is not None:
            head_input = self.head(head_input)
        return head_input
def load_nam_weights(nam_path, model, head_scale):
    with open(nam_path) as f:
        nam = json.load(f)
    w = torch.from_numpy(np.array(nam["weights"], dtype=np.float32))
    param_tensors = []
    for name, param in model.named_parameters():
        if any(k in name for k in ['head', 'conv', 'mixer', 'rechannel', '1x1']):
            param_tensors.append(param)
    ptr = 0
    for pt in param_tensors:
        n = pt.numel()
        pt.data.copy_(w[ptr:ptr+n].reshape(pt.shape))
        ptr += n
    return model
def expand_to_2channel(p1_sd, p2_model):
    p2_sd = p2_model.state_dict()
    for key in p2_sd:
        if key not in p1_sd:
            p2_sd[key].zero_()
            continue
        w1 = p1_sd[key]
        w2 = p2_sd[key]
        if w1.shape == w2.shape:
            p2_sd[key] = w1.clone()
        elif w1.dim() == 3 and w2.dim() == 3 and w2.shape[1] > w1.shape[1]:
            p2_sd[key][:, :1] = w1
            p2_sd[key][:, 1:] = 0
        elif w1.dim() == 1 and w2.dim() == 1 and w1.numel() < w2.numel():
            p2_sd[key][:] = 0
            p2_sd[key][:w1.shape[0]] = w1
        else:
            p2_sd[key] = w1.clone()[:w2.shape[0]] if w1.numel() >= w2.numel() else 0
    return p2_sd
class MultiRateDataset(Dataset):
    def __init__(self, di_path, cap_path, receptive_field, ny=8192,
                 train_ratio=0.9, seed=42, cap_resampled=None):
        self.rng = np.random.RandomState(seed)
        self.nx = receptive_field
        self.ny = ny
        self.sample_length = self.nx + ny - 1
        di_orig, di_rate = librosa.load(str(di_path), sr=None, mono=True)
        cap_orig, cap_rate = librosa.load(str(cap_path), sr=None, mono=True)
        min_len = min(len(di_orig), len(cap_orig))
        di_orig = di_orig[:min_len]
        cap_orig = cap_orig[:min_len]
        self.precomputed = {}
        for rate in TRAIN_RATES:
            if cap_resampled is not None and rate in cap_resampled:
                di_r = get_resampled_di(di_path, di_rate, rate)
                cap_r = cap_resampled[rate]
                ml = min(len(di_r), len(cap_r))
                di_r = di_r[:ml]
                cap_r = cap_r[:ml]
            else:
                di_r = get_resampled_di(di_path, di_rate, rate)
                cap_r = resample_audio(cap_orig, cap_rate, rate)
                di_r = di_r[:len(cap_r)] if len(di_r) > len(cap_r) else di_r
                cap_r = cap_r[:len(di_r)] if len(cap_r) > len(di_r) else cap_r
            self.precomputed[rate] = (di_r, cap_r)
        self.samples_per_rate = {}
        for rate, (di_r, cap_r) in self.precomputed.items():
            n = (len(di_r) - self.nx + 1) // self.ny
            self.samples_per_rate[rate] = max(n, 1)
        self.total_samples = sum(self.samples_per_rate.values())
        all_idx = list(range(self.total_samples))
        self.rng.shuffle(all_idx)
        split = int(len(all_idx) * train_ratio)
        self.train_indices = sorted(all_idx[:split])
        self.val_indices = sorted(all_idx[split:])
    def _get_item(self, idx):
        cumulative = 0
        for rate, count in self.samples_per_rate.items():
            if idx < cumulative + count:
                local_idx = idx - cumulative
                di_r, cap_r = self.precomputed[rate]
                i = local_idx * self.ny
                j = i + self.nx - 1
                x_chunk = di_r[i:i + self.sample_length]
                y_chunk = cap_r[j:j + self.ny]
                if len(x_chunk) < self.sample_length:
                    pad = self.sample_length - len(x_chunk)
                    x_chunk = np.pad(x_chunk, (0, pad))
                    y_chunk = np.pad(y_chunk, (0, self.ny - len(y_chunk)))
                rate_norm = (rate - 48000) / MAX_RATE
                rate_ch = np.full_like(x_chunk, rate_norm)
                x_2ch = np.stack([x_chunk, rate_ch], axis=0)
                return (torch.from_numpy(x_2ch).float(),
                        torch.from_numpy(y_chunk).float())
            cumulative += count
        raise IndexError
    def __getitem__(self, idx):
        return self._get_item(idx)
    def __len__(self):
        return self.total_samples
@torch.no_grad()
def validate(model, loader, device, steps=None):
    model.eval()
    sum_se, sum_st = 0.0, 0.0
    for i, (x, y) in enumerate(loader):
        if steps and i >= steps:
            break
        x, y = x.to(device), y.to(device)
        pred = model(x).squeeze(1)
        sum_se += torch.sum((pred - y) ** 2).item()
        sum_st += torch.sum(y ** 2).item()
    return sum_se / (sum_st + 1e-8)
def export_model(model, layers_configs, head_scale, output_path, basename,
                 metadata=None, sample_rate=48000):
    weights_list = []
    for name, param in model.named_parameters():
        if any(k in name for k in ['head', 'conv', 'mixer', 'rechannel', '1x1',
                                    'rate_condition']):
            weights_list.append(param.data.cpu().numpy().ravel())
    weights = np.concatenate(weights_list)
    weights = np.append(weights, np.array([head_scale]))
    config = {
        "layers": [dict(lc) for lc in layers_configs],
        "head": None,
        "head_scale": head_scale,
        "rate_condition_hidden": model.rate_condition.fc1.out_features,
        "rate_condition_num_biases": model.rate_condition.fc2.out_features,
    }
    meta = {"date": time.strftime("%Y-%m-%dT%H:%M:%S"),
            "sample_rate": sample_rate, "loudness": None}
    if metadata:
        meta.update(metadata)
    nam_data = {
        "version": "0.5.4", "metadata": meta,
        "architecture": "WaveNet", "config": config,
        "weights": weights.tolist(),
    }
    os.makedirs(output_path, exist_ok=True)
    filepath = os.path.join(output_path, f"{basename}.namos")
    with open(filepath, "w") as f:
        json.dump(nam_data, f, indent=2)
    return filepath
def make_1ch_config():
    return [
        dict(input_size=1, condition_size=1, channels=16, head_size=8,
             kernel_size=3, dilations=[1,2,4,8,16,32,64,128,256,512],
             activation="Tanh", gated=False, head_bias=False),
        dict(input_size=16, condition_size=1, channels=8, head_size=1,
             kernel_size=3, dilations=[1,2,4,8,16,32,64,128,256,512],
             activation="Tanh", gated=False, head_bias=True),
    ]
def make_2ch_config():
    return [
        dict(input_size=2, condition_size=2, channels=16, head_size=8,
             kernel_size=3, dilations=[1,2,4,8,16,32,64,128,256,512],
             activation="Tanh", gated=False, head_bias=False),
        dict(input_size=16, condition_size=2, channels=8, head_size=1,
             kernel_size=3, dilations=[1,2,4,8,16,32,64,128,256,512],
             activation="Tanh", gated=False, head_bias=True),
    ]
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
pdir = Path('pretrained')
cdir = Path('captures')
odir = Path('output')
di_path = next(Path('di').glob('*.wav'))
p1_cfg = make_1ch_config()
p2_cfg = make_2ch_config()
dummy = WaveNetCore(p1_cfg, head_scale=0.02, use_rate_condition=False)
rf = dummy.receptive_field
del dummy
cap_files = sorted(cdir.rglob('*.wav'))
cap_map = {}
for p in cap_files:
    stem = p.stem
    clean = re.sub(r'_in_[\d.]+_out_-?[\d.]+', '', stem)
    cap_map[clean] = p
pt_files = sorted(pdir.rglob('*.nam'))
print(f"Models: {len(pt_files)} | Captures: {len(cap_files)}")
di_orig, di_rate = librosa.load(str(di_path), sr=None, mono=True)
for rate in TRAIN_RATES:
    get_resampled_di(di_path, di_rate, rate)
cap_resampled = {}
for clean_stem, cap_path in cap_map.items():
    cap_orig, cap_rate = librosa.load(str(cap_path), sr=None, mono=True)
    cap_resampled[clean_stem] = {}
    for rate in TRAIN_RATES:
        cap_resampled[clean_stem][rate] = resample_audio(cap_orig, cap_rate, rate)
successes, failures = 0, 0
epochs = epochs_w.value
patience = patience_w.value
lr = lr_w.value
batch_size = batch_w.value
ny = ny_w.value
val_freq = val_freq_w.value
steps_per_epoch = steps_per_epoch_w.value
for pt_path in tqdm(pt_files, desc='Models'):
    clean_stem = pt_path.stem
    if clean_stem not in cap_map:
        print(f"[SKIP] {clean_stem} — no capture")
        failures += 1
        continue
    cap_path = cap_map[clean_stem]
    print(f"\n🎸 {clean_stem}")
    model1 = WaveNetCore(p1_cfg, head_scale=0.02, use_rate_condition=False)
    try:
        model1 = load_nam_weights(pt_path, model1, 0.02)
    except Exception as e:
        print(f"  ✗ Load failed: {e}")
        failures += 1
        continue
    p1_sd = model1.state_dict()
    model2 = WaveNetCore(p2_cfg, head_scale=0.02, use_rate_condition=True)
    p2_sd = expand_to_2channel(p1_sd, model2)
    model2.load_state_dict(p2_sd)
    model2.to(device)
    del model1
    full_ds = MultiRateDataset(di_path, cap_path, rf, ny=ny,
                               cap_resampled=cap_resampled.get(clean_stem))
    train_ds = torch.utils.data.Subset(full_ds, full_ds.train_indices)
    val_ds = torch.utils.data.Subset(full_ds, full_ds.val_indices)
    train_loader = DataLoader(train_ds, batch_size=batch_size,
                              shuffle=True, drop_last=True, num_workers=0)
    val_loader = DataLoader(val_ds, batch_size=batch_size,
                             drop_last=False, num_workers=0)
    optimizer = torch.optim.Adam(model2.parameters(), lr=lr)
    best_esr = float('inf')
    best_state = None
    no_improve = 0
    epoch_pbar = tqdm(range(epochs), desc='Training', leave=False)
    for epoch in epoch_pbar:
        model2.train()
        sum_se, sum_st = 0.0, 0.0
        for i, (x, y) in enumerate(train_loader):
            if steps_per_epoch > 0 and i >= steps_per_epoch:
                break
            x, y = x.to(device), y.to(device)
            optimizer.zero_grad()
            pred = model2(x).squeeze(1)
            loss = F.mse_loss(pred, y)
            loss.backward()
            optimizer.step()
            sum_se += torch.sum((pred - y) ** 2).item()
            sum_st += torch.sum(y ** 2).item()
        train_esr = sum_se / (sum_st + 1e-8)
        if (epoch + 1) % val_freq == 0 or epoch == 0:
            val_esr = validate(model2, val_loader, device, steps=50)
            epoch_pbar.set_postfix({'train': f'{train_esr:.4f}', 'val': f'{val_esr:.4f}'})
            if val_esr < best_esr:
                best_esr = val_esr
                best_state = {k: v.cpu().clone()
                              for k, v in model2.state_dict().items()}
                no_improve = 0
            else:
                no_improve += 1
            if patience > 0 and no_improve >= patience:
                print(f"  Early stop at epoch {epoch+1}")
                break
    if best_state:
        model2.load_state_dict(best_state)
    model2.to('cpu')
    cap_stem = cap_path.stem
    cap_levels = re.search(r'_in_(-?[\d.]+)_out_(-?[\d.]+)', cap_stem)
    meta = {"training_esr": best_esr,
            "pretrained": pt_path.name,
            "architecture": "STANDARD"}
    if cap_levels:
        meta["input_level_dbu"] = float(cap_levels.group(1))
        meta["output_level_dbu"] = float(cap_levels.group(2))
    try:
        export_model(model2, p2_cfg, 0.02, str(odir),
                     clean_stem, metadata=meta)
        print(f"  ✓ Saved {clean_stem}.namos")
        successes += 1
    except Exception as e:
        print(f"  ✗ Export failed: {e}")
        failures += 1
print(f"\n{'='*70}")
print(f"Done: {successes} succeeded, {failures} failed")


In [ ]:
import glob
from google.colab import files
namos_files = glob.glob('output/*.namos')
print(f"Found {len(namos_files)} models:")
for f in namos_files:
    print(f"  {os.path.basename(f)}")
if namos_files:
    print("\n📦 Zipping for download...")
    !zip -q NamOS_models.zip output/*.namos
    files.download('NamOS_models.zip')
else:
    print("No models found in output/")
